# StratoMesh — Orthomosaic Segmentation

Segments a **true-colour RGB GeoTIFF** (orthomosaic) into the five classes needed for 3D reconstruction:
**Buildings · Roads · Cars · Trees · Vegetation**

Uses [`samgeo`](https://samgeo.gishub.org/) which combines:
- **GroundingDINO** — open-vocabulary text-to-box detector
- **SAM ViT-H** — Meta's Segment Anything Model (largest checkpoint) for precise masks

**Runtime:** Change to `T4 GPU` in *Runtime → Change runtime type* before running.

In [ ]:
# ── 0. Verify GPU ─────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU.')
print('GPU:', result.stdout.strip())

import torch
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
# segment-geospatial bundles samgeo + LangSAM + GroundingDINO + SAM 2
%pip install -q segment-geospatial
%pip install -q rasterio matplotlib numpy Pillow

# Restart runtime once after install so imports resolve correctly
print('Install complete. If this is the first run, go to Runtime → Restart session, then re-run from cell 0.')

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import rasterio
from rasterio.transform import from_bounds
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Ready.')

In [ ]:
# ── 3. Upload your GeoTIFF ────────────────────────────────────────────────────
from google.colab import files

print('Select your .tif orthomosaic file:')
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError('No file uploaded.')

TIF_PATH = list(uploaded.keys())[0]
print(f'\nUploaded: {TIF_PATH}  ({os.path.getsize(TIF_PATH)/1e6:.1f} MB)')

In [ ]:
# ── 4. Inspect & preview ─────────────────────────────────────────────────────
with rasterio.open(TIF_PATH) as src:
    META = {
        'bands':  src.count,
        'width':  src.width,
        'height': src.height,
        'crs':    str(src.crs),
        'dtype':  src.dtypes[0],
        'bounds': src.bounds,
        'transform': src.transform,
    }
    # Read R, G, B (bands 1-3)
    rgb = np.stack([src.read(1), src.read(2), src.read(3)], axis=2).astype(np.uint8)
    ALPHA = src.read(4) if src.count >= 4 else None
    VALID_MASK = (ALPHA > 0) if ALPHA is not None else np.ones(rgb.shape[:2], bool)

print('── Image info ──────────────────────────────')
for k, v in META.items():
    print(f'  {k:<12}: {v}')

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(rgb)
ax.set_title(f'Orthomosaic preview  |  {META["width"]} x {META["height"]} px', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. Handle large files ─────────────────────────────────────────────────────
# LangSAM / GroundingDINO work best up to ~2048 px on one side.
# We write a working copy at capped resolution — the geo-transform is adjusted
# so exported masks stay georeferenced.

MAX_DIM = 2048

h, w = rgb.shape[:2]
scale = min(1.0, MAX_DIM / max(h, w))

if scale < 1.0:
    new_w, new_h = int(w * scale), int(h * scale)
    print(f'Downsampling {w}x{h}  →  {new_w}x{new_h}  (scale {scale:.3f})')
    rgb_work = np.array(Image.fromarray(rgb).resize((new_w, new_h), Image.LANCZOS))
else:
    new_w, new_h = w, h
    rgb_work = rgb
    print(f'Image fits within {MAX_DIM}px — no downscaling needed.')

# Save working RGB TIF (samgeo needs a file path, not an array)
WORK_TIF = str(OUTPUT_DIR / 'work_rgb.tif')
b = META['bounds']
work_transform = from_bounds(b.left, b.bottom, b.right, b.top, new_w, new_h)

with rasterio.open(
    WORK_TIF, 'w',
    driver='GTiff', count=3, dtype='uint8',
    width=new_w, height=new_h,
    crs=META['crs'], transform=work_transform,
) as dst:
    for i in range(3):
        dst.write(rgb_work[:, :, i], i + 1)

print(f'Working TIF saved: {WORK_TIF}')

In [ ]:
# ── 6. Load LangSAM (SAM ViT-H + GroundingDINO) ─────────────────────────────
# First run downloads ~2.5 GB of model weights — takes 2-3 min on T4.
# Do NOT pass model_type='sam2' — it hits an unresolved import in current samgeo.
from samgeo.text_sam import LangSAM

sam = LangSAM()   # defaults to SAM ViT-H (largest, most accurate checkpoint)
print('LangSAM loaded.')

In [ ]:
# ── 7. Define categories & run text-prompted segmentation ─────────────────────
#
# box_threshold  — confidence required for GroundingDINO to propose a box
# text_threshold — minimum text-image similarity to keep a box
#
# Lower these if categories are being missed; raise them to cut false positives.

CATEGORIES = {
    'building':   {'box_thresh': 0.30, 'text_thresh': 0.25, 'min_px': 200,  'color': '#E74C3C'},
    'road':       {'box_thresh': 0.25, 'text_thresh': 0.20, 'min_px': 150,  'color': '#95A5A6'},
    'car':        {'box_thresh': 0.22, 'text_thresh': 0.18, 'min_px': 20,   'color': '#3498DB'},
    'tree':       {'box_thresh': 0.28, 'text_thresh': 0.22, 'min_px': 50,   'color': '#27AE60'},
    'vegetation': {'box_thresh': 0.22, 'text_thresh': 0.18, 'min_px': 100,  'color': '#2ECC71'},
}

results = {}   # category -> dict with masks / boxes / labels

for cat, cfg in CATEGORIES.items():
    print(f'\n── {cat.upper()} ─────────────────────────────')
    sam.predict(
        image=WORK_TIF,
        text_prompt=cat,
        box_threshold=cfg['box_thresh'],
        text_threshold=cfg['text_thresh'],
    )

    n_boxes = len(sam.boxes) if sam.boxes is not None else 0
    print(f'  Detected: {n_boxes} instances')

    if n_boxes == 0:
        print('  Nothing found — try lowering box_threshold.')
        results[cat] = None
        continue

    # ── Binary mask (1=class, 0=other) ────────────────────────────────────
    binary_path = str(OUTPUT_DIR / f'mask_{cat}_binary.tif')
    sam.save_masks(
        output=binary_path,
        foreground=True,
        dtype='uint8',
        min_size=cfg['min_px'],
        crs=META['crs'],
        transform=work_transform,
    )

    # ── Instance mask (unique ID per object) ───────────────────────────────
    instance_path = str(OUTPUT_DIR / f'mask_{cat}_instances.tif')
    sam.save_masks(
        output=instance_path,
        foreground=False,
        dtype='uint16',
        min_size=cfg['min_px'],
        crs=META['crs'],
        transform=work_transform,
    )

    results[cat] = {
        'n_instances': n_boxes,
        'binary_tif':  binary_path,
        'instance_tif': instance_path,
        'boxes': sam.boxes.tolist() if hasattr(sam.boxes, 'tolist') else sam.boxes,
        'scores': sam.logits.tolist() if hasattr(sam, 'logits') and sam.logits is not None else [],
    }
    print(f'  Saved: {binary_path}')
    print(f'         {instance_path}')

print('\n── Done ──────────────────────────────────────')

In [ ]:
# ── 8. Build composite label map ──────────────────────────────────────────────
# Priority order (later overwrites earlier if pixels overlap):
#   vegetation < tree < road < building < car
PRIORITY = ['vegetation', 'tree', 'road', 'building', 'car']
CLASS_IDS = {cat: i + 1 for i, cat in enumerate(PRIORITY)}  # 1-indexed

label_map = np.zeros((new_h, new_w), dtype=np.uint8)

for cat in PRIORITY:
    if results.get(cat) is None:
        continue
    with rasterio.open(results[cat]['binary_tif']) as src:
        mask = src.read(1)
    label_map[mask > 0] = CLASS_IDS[cat]

# Save composite GeoTIFF
COMPOSITE_PATH = str(OUTPUT_DIR / 'label_map.tif')
with rasterio.open(
    COMPOSITE_PATH, 'w',
    driver='GTiff', count=1, dtype='uint8',
    width=new_w, height=new_h,
    crs=META['crs'], transform=work_transform,
) as dst:
    dst.write(label_map, 1)

print(f'Composite label map saved: {COMPOSITE_PATH}')
unique, counts = np.unique(label_map, return_counts=True)
id_to_name = {0: 'background', **{v: k for k, v in CLASS_IDS.items()}}
print('\nPixel counts:')
for cls_id, cnt in zip(unique, counts):
    pct = cnt / label_map.size * 100
    print(f'  {id_to_name.get(cls_id, cls_id):<12} (id={cls_id}): {cnt:>8,} px  ({pct:.1f}%)')

In [ ]:
# ── 9. Visualize ─────────────────────────────────────────────────────────────
PALETTE = {
    0: (20,  20,  20),   # background
    1: (46,  204, 113),  # vegetation — green
    2: (39,  174, 96),   # tree — dark green
    3: (149, 165, 166),  # road — grey
    4: (231, 76,  60),   # building — red
    5: (52,  152, 219),  # car — blue
}

colour_map = np.zeros((new_h, new_w, 3), dtype=np.uint8)
for cls_id, colour in PALETTE.items():
    colour_map[label_map == cls_id] = colour

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

axes[0].imshow(rgb_work)
axes[0].set_title('Original orthomosaic', fontsize=13)
axes[0].axis('off')

axes[1].imshow(rgb_work)
axes[1].imshow(colour_map, alpha=0.55)
axes[1].set_title('Segmentation overlay', fontsize=13)
axes[1].axis('off')

legend_patches = [
    mpatches.Patch(color=np.array(c)/255, label=id_to_name[i])
    for i, c in PALETTE.items() if i > 0
]
axes[1].legend(handles=legend_patches, loc='lower right',
               fontsize=11, framealpha=0.85)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segmentation_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/segmentation_overlay.png')

In [ ]:
# ── 10. Per-category preview ──────────────────────────────────────────────────
found = [c for c in PRIORITY if results.get(c) is not None]
fig, axes = plt.subplots(1, len(found), figsize=(6 * len(found), 5))
if len(found) == 1:
    axes = [axes]

for ax, cat in zip(axes, found):
    with rasterio.open(results[cat]['binary_tif']) as src:
        bmask = src.read(1)
    overlay = rgb_work.copy()
    colour = np.array([int(CATEGORIES[cat]['color'].lstrip('#')[i:i+2], 16) for i in (0,2,4)])
    overlay[bmask > 0] = (overlay[bmask > 0] * 0.4 + colour * 0.6).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f'{cat}  ({results[cat]["n_instances"]} instances)', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_category.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. Save summary JSON ─────────────────────────────────────────────────────
summary = {
    'source_tif': TIF_PATH,
    'resolution': f'{new_w}x{new_h}',
    'scale_factor': round(scale, 4),
    'crs': META['crs'],
    'categories': {
        cat: {
            'instances': res['n_instances'],
            'binary_tif': res['binary_tif'],
            'instance_tif': res['instance_tif'],
        } if res else {'instances': 0}
        for cat, res in results.items()
    },
    'label_map': COMPOSITE_PATH,
    'class_ids': CLASS_IDS,
}

SUMMARY_PATH = OUTPUT_DIR / 'segmentation_summary.json'
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Summary saved: {SUMMARY_PATH}')
print(json.dumps({c: summary['categories'][c]['instances'] for c in PRIORITY}, indent=2))

In [ ]:
# ── 12. Download all outputs ──────────────────────────────────────────────────
import shutil

ZIP_PATH = 'stratomesh_segmentation.zip'
shutil.make_archive('stratomesh_segmentation', 'zip', OUTPUT_DIR)
print(f'Archive: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)')

files.download(ZIP_PATH)
print('\nContents:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1e3:.0f} KB)')

## Output files

| File | Description |
|---|---|
| `label_map.tif` | Single-band GeoTIFF — each pixel has a class ID (0=bg, 1=vegetation, 2=tree, 3=road, 4=building, 5=car) |
| `mask_<cat>_binary.tif` | Per-category binary mask (1=class, 0=other) |
| `mask_<cat>_instances.tif` | Per-category instance mask — each object gets a unique integer ID |
| `segmentation_overlay.png` | Visual overlay for inspection |
| `segmentation_summary.json` | Instance counts + file paths |

## Tuning tips

- **Missing detections** — lower `box_threshold` by 0.05 at a time
- **Too many false positives** — raise `text_threshold` or increase `min_px`
- **Cars on rooftops being called buildings** — add a negative prompt: change `'car'` → `'car . vehicle'` and `'building'` → `'building . roof'`
- **Large file runs out of VRAM** — reduce `MAX_DIM` from 2048 to 1024 in cell 5

## Next step (Stage 2)

Feed `mask_building_instances.tif` into StratoMesh:
each building polygon → OSM height lookup or manual height override → `THREE.ExtrudeGeometry` block.